In [ ]:
# One-time: the project was renamed. If an older Drive folder exists, rename it so cached data and weights are kept.
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    _old, _new = '/content/drive/MyDrive/' + 'exo' + 'lens', '/content/drive/MyDrive/anantara'
    if os.path.isdir(_old) and not os.path.exists(_new): os.rename(_old, _new); print('Renamed old Drive folder ->', _new)
except Exception: pass

# Model B – 1D CNN on Kepler + TESS light curves
Runtime → **T4 GPU**. Resumable: results are cached on Google Drive, so after a disconnect just re-run all cells.

In [ ]:
!pip -q install lightkurve
import os, time, io, requests, warnings
import numpy as np, pandas as pd, lightkurve as lk
warnings.filterwarnings('ignore')
from google.colab import drive; drive.mount('/content/drive')
ROOT='/content/drive/MyDrive/anantara'; os.makedirs(ROOT, exist_ok=True)
import torch; print('GPU:', torch.cuda.is_available())

In [ ]:
# ---- SETTINGS: shrink these for a quick test ----
N_PER_CLASS_KEPLER = 150
N_PER_CLASS_TESS   = 150
DOWNLOAD_BUDGET    = 75   # minutes; loop stops itself
TRAIN_BUDGET       = 20   # minutes
BINS, SEED         = 201, 42

## 1. Labels from the NASA Exoplanet Archive (Kepler = KOI table, TESS = TOI table)

In [ ]:
def archive(q):
    r=requests.get('https://exoplanetarchive.ipac.caltech.edu/TAP/sync',params={'query':q,'format':'csv'}); return pd.read_csv(io.StringIO(r.text))
koi=archive("select kepid,koi_disposition,koi_period,koi_time0bk,koi_duration from cumulative where koi_period<40 and koi_disposition in ('CONFIRMED','FALSE POSITIVE')").dropna()
koi=pd.DataFrame({'mission':'Kepler','star':koi.kepid.astype(int),'P':koi.koi_period,'t0':koi.koi_time0bk,'dur_hr':koi.koi_duration,'label':(koi.koi_disposition=='CONFIRMED').astype(int)})
toi=archive("select tid,tfopwg_disp,pl_orbper,pl_tranmid,pl_trandurh from toi where tfopwg_disp in ('CP','KP','FP','FA')").dropna()
toi=pd.DataFrame({'mission':'TESS','star':toi.tid.astype(int),'P':toi.pl_orbper,'t0':toi.pl_tranmid-2457000,'dur_hr':toi.pl_trandurh,'label':toi.tfopwg_disp.isin(['CP','KP']).astype(int)})  # BJD -> BTJD
toi=toi[toi.P<40]
def pick(df,n): return pd.concat([g.drop_duplicates('star').sample(min(len(g.drop_duplicates('star')),n),random_state=SEED) for _,g in df.groupby('label')])
table=pd.concat([pick(koi,N_PER_CLASS_KEPLER),pick(toi,N_PER_CLASS_TESS)]).reset_index(drop=True)
print(table.groupby(['mission','label']).size())

## 2. Download + turn each star into a 2×201 array (global view + zoomed transit view)

In [ ]:
def binned(ph,f,lo,hi,nb):
    idx=np.digitize(ph,np.linspace(lo,hi,nb+1))-1; out=np.full(nb,np.nan)
    for i in range(nb):
        m=idx==i
        if m.any(): out[i]=np.median(f[m])
    ok=~np.isnan(out)
    if ok.sum()<nb//2: return None
    out[~ok]=np.interp(np.where(~ok)[0],np.where(ok)[0],out[ok]); return out

def make_views(lc,P,t0,dur_hr):
    dur=dur_hr/24; cad=np.median(np.diff(lc.time.value))
    win=int(np.clip(4*dur,0.75,3)/cad)//2*2+1                       # odd window, ~4x transit length
    mask=lc.create_transit_mask(period=P,transit_time=t0,duration=dur*2)
    flat=lc.flatten(window_length=win,mask=mask).remove_outliers(sigma_upper=4,sigma_lower=20)
    ph=((flat.time.value-t0+0.5*P)%P)-0.5*P
    f=flat.flux.value/np.nanmedian(flat.flux.value)-1
    g=binned(ph,f,-P/2,P/2,BINS); l=binned(ph,f,-3*dur,3*dur,BINS)
    if g is None or l is None: return None
    s=abs(min(g.min(),l.min()))+1e-8
    return np.stack([g/s,l/s]).astype(np.float32)

CACHE=f'{ROOT}/views_kepler_tess.npz'
if os.path.exists(CACHE):
    d=np.load(CACHE,allow_pickle=True); done=dict(zip(d['keys'].tolist(),list(d['X'])))
else: done={}
def save(): np.savez(CACHE,keys=np.array(list(done.keys())),X=np.stack(list(done.values())))
t_start=time.time()
for n,r in enumerate(table.itertuples()):
    key=f'{r.mission}_{r.star}'
    if key in done: continue
    if (time.time()-t_start)/60>DOWNLOAD_BUDGET: print('download budget reached'); break
    try:
        if r.mission=='Kepler': sr=lk.search_lightcurve(f'KIC {r.star}',mission='Kepler',author='Kepler',cadence='long')[:6]
        else:                   sr=lk.search_lightcurve(f'TIC {r.star}',mission='TESS',author='SPOC',exptime=120)[:4]
        lc=sr.download_all().stitch().remove_nans()
        v=make_views(lc,r.P,r.t0,r.dur_hr)
        if v is not None: done[key]=v
    except Exception: pass
    if n%25==0 and done: save(); print(n,len(done),f'{(time.time()-t_start)/60:.1f} min')
save(); print('saved',len(done))

## 3. Dataset (split by star)

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
d=np.load(CACHE,allow_pickle=True); keys,X=d['keys'].tolist(),d['X']
lab={f'{r.mission}_{r.star}':r.label for r in table.itertuples()}
y=np.array([lab[k] for k in keys])
Xtr,Xt,ytr,yt=train_test_split(X,y,test_size=0.3,stratify=y,random_state=SEED)
Xva,Xte,yva,yte=train_test_split(Xt,yt,test_size=0.5,stratify=yt,random_state=SEED)
print(Xtr.shape,Xva.shape,Xte.shape,'planet fraction',y.mean().round(2))
class LC(Dataset):
    def __init__(s,X,y,aug=False): s.X,s.y,s.aug=X,y,aug
    def __len__(s): return len(s.X)
    def __getitem__(s,i):
        x=s.X[i].copy()
        if s.aug:
            if np.random.rand()<.5: x=x[:,::-1].copy()
            x+=np.random.normal(0,.03,x.shape).astype(np.float32)
        return torch.from_numpy(x),torch.tensor(s.y[i],dtype=torch.float32)
dl_tr=DataLoader(LC(Xtr,ytr,True),batch_size=32,shuffle=True); dl_va=DataLoader(LC(Xva,yva),batch_size=64); dl_te=DataLoader(LC(Xte,yte),batch_size=64)

## 4. The 1D CNN

In [ ]:
import torch.nn as nn, torch.nn.functional as F
class TransitCNN1D(nn.Module):
    def __init__(s):
        super().__init__()
        blk=lambda i,o: nn.Sequential(nn.Conv1d(i,o,5,padding=2),nn.BatchNorm1d(o),nn.ReLU(),nn.MaxPool1d(2))
        s.features=nn.Sequential(blk(2,16),blk(16,32),blk(32,64),nn.Conv1d(64,64,5,padding=2),nn.ReLU())
        s.head=nn.Sequential(nn.Dropout(.4),nn.Linear(64,1))
    def forward(s,x): return s.head(s.features(x).mean(-1)).squeeze(-1)
dev='cuda' if torch.cuda.is_available() else 'cpu'
model=TransitCNN1D().to(dev)
opt=torch.optim.AdamW(model.parameters(),1e-3,weight_decay=1e-2)
lossf=nn.BCEWithLogitsLoss(pos_weight=torch.tensor((1-ytr.mean())/ytr.mean(),dtype=torch.float32).to(dev))

## 5. Train (stops itself at the time budget; best model saved to Drive)

In [ ]:
from sklearn.metrics import roc_auc_score
def evaluate(dl):
    model.eval(); P=[];Y=[]
    with torch.no_grad():
        for xb,yb in dl: P+=torch.sigmoid(model(xb.to(dev))).cpu().tolist(); Y+=yb.tolist()
    return roc_auc_score(Y,P), float(np.mean((np.array(P)>.5)==np.array(Y)))
best,t0=0,time.time()
for ep in range(100):
    model.train()
    for xb,yb in dl_tr:
        xb,yb=xb.to(dev),yb.to(dev); opt.zero_grad(); loss=lossf(model(xb),yb); loss.backward(); opt.step()
    auc,acc=evaluate(dl_va); print(f'ep {ep:02d} loss {loss.item():.3f} val AUC {auc:.3f} acc {acc:.3f}')
    if auc>best: best=auc; torch.save(model.state_dict(),f'{ROOT}/model_b_1d.pt')
    if (time.time()-t0)/60>TRAIN_BUDGET: print('train budget reached'); break
model.load_state_dict(torch.load(f'{ROOT}/model_b_1d.pt')); print('TEST AUC, acc:',evaluate(dl_te))

## 6. Grad-CAM: *where in the light curve* did the CNN look?

In [ ]:
import matplotlib.pyplot as plt
def gradcam(x):
    model.eval(); f=model.features(x[None].to(dev)); f.retain_grad()
    out=model.head(f.mean(-1)).squeeze(-1); out.backward()
    cam=F.relu((f.grad.mean(-1,keepdim=True)*f).sum(1))[0].detach().cpu().numpy()
    cam=np.interp(np.linspace(0,len(cam)-1,BINS),np.arange(len(cam)),cam); return cam/(cam.max()+1e-8), torch.sigmoid(out).item()
i=int(np.where(yte==1)[0][0]); x=torch.from_numpy(Xte[i]); cam,p=gradcam(x)
fig,ax=plt.subplots(1,2,figsize=(11,3))
for a,k,t in zip(ax,[0,1],['Global view','Local view']):
    a.plot(x[k],'k.',ms=3); a.set_title(f'{t} (planet prob {p:.2f})'); a2=a.twinx(); a2.fill_between(range(BINS),cam,color='orange',alpha=.35)
plt.tight_layout(); plt.show()